# 🎓 Лабораторная работа 12. Fine-tuning, LoRA и QLoRA

Цель: понять LoRA не как готовую библиотечную команду, а как небольшое обучаемое дополнение к замороженному Linear-слою.

Мы реализуем учебную LoRA вручную на PyTorch.


# 1. Импорт

In [ ]:
import copy

import torch
import torch.nn as nn

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

print("PyTorch:", torch.__version__)

# 2. Обычный Linear-слой

In [ ]:
INPUT_DIM = 16
OUTPUT_DIM = 16

base_layer = nn.Linear(
    INPUT_DIM,
    OUTPUT_DIM,
)

print(base_layer)
print("Weight shape:", base_layer.weight.shape)
print("Bias shape:", base_layer.bias.shape)

# 3. Сколько параметров у Base Layer

In [ ]:
base_parameters = sum(
    parameter.numel()
    for parameter in base_layer.parameters()
)

print("Base parameters:", base_parameters)

# 4. Замораживаем Base weights

In [ ]:
for parameter in base_layer.parameters():
    parameter.requires_grad = False

for name, parameter in base_layer.named_parameters():
    print(
        name,
        "requires_grad =",
        parameter.requires_grad,
    )

# 5. Реализуем LoRA Linear

In [ ]:
class LoRALinear(nn.Module):
    def __init__(
        self,
        base_layer,
        rank=4,
        alpha=8.0,
    ):
        super().__init__()

        self.base = base_layer
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank

        for parameter in self.base.parameters():
            parameter.requires_grad = False

        self.lora_A = nn.Parameter(
            torch.randn(
                rank,
                base_layer.in_features,
            ) * 0.01
        )

        self.lora_B = nn.Parameter(
            torch.zeros(
                base_layer.out_features,
                rank,
            )
        )

    def forward(self, x):
        base_output = self.base(x)

        lora_update = (
            x
            @ self.lora_A.T
            @ self.lora_B.T
        )

        return (
            base_output
            + self.scale * lora_update
        )


lora_layer = LoRALinear(
    base_layer,
    rank=4,
    alpha=8.0,
)

print(lora_layer)

# 6. Shapes A и B

In [ ]:
print("A:", lora_layer.lora_A.shape)
print("B:", lora_layer.lora_B.shape)

print(
    "Delta W theoretical shape:",
    (
        lora_layer.lora_B
        @ lora_layer.lora_A
    ).shape,
)

# 7. Какие параметры обучаются

In [ ]:
for name, parameter in lora_layer.named_parameters():
    print(
        name,
        parameter.shape,
        "requires_grad =",
        parameter.requires_grad,
    )

# 8. Сравниваем количество параметров

In [ ]:
total_parameters = sum(
    parameter.numel()
    for parameter in lora_layer.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in lora_layer.parameters()
    if parameter.requires_grad
)

print("Всего параметров:", total_parameters)
print("Обучаемых:", trainable_parameters)

print(
    "Trainable %:",
    100 * trainable_parameters / total_parameters,
)

# 9. Почему B начинается с нулей

In [ ]:
x = torch.randn(3, INPUT_DIM)

with torch.no_grad():
    base_output = base_layer(x)
    lora_output = lora_layer(x)

print(
    "В начале outputs совпадают:",
    torch.allclose(
        base_output,
        lora_output,
        atol=1e-6,
    )
)

Поскольку `B` начинается с нулей, LoRA-ветка сначала почти ничего не добавляет. Модель стартует с поведения Base Layer.


# 10. Сохраняем Base weights до обучения

In [ ]:
base_weight_before = (
    lora_layer.base.weight
    .detach()
    .clone()
)

base_bias_before = (
    lora_layer.base.bias
    .detach()
    .clone()
)

A_before = (
    lora_layer.lora_A
    .detach()
    .clone()
)

B_before = (
    lora_layer.lora_B
    .detach()
    .clone()
)

# 11. Учебный Dataset

In [ ]:
features = torch.randn(
    128,
    INPUT_DIM,
)

target_projection = torch.randn(
    INPUT_DIM,
    OUTPUT_DIM,
)

targets = features @ target_projection

print(features.shape)
print(targets.shape)

# 12. Optimizer видит только trainable parameters

In [ ]:
optimizer = torch.optim.Adam(
    [
        parameter
        for parameter in lora_layer.parameters()
        if parameter.requires_grad
    ],
    lr=0.03,
)

loss_function = nn.MSELoss()

print(optimizer)

# 13. Обучаем только LoRA

In [ ]:
history = []

for epoch in range(200):
    prediction = lora_layer(features)
    loss = loss_function(
        prediction,
        targets,
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    history.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(
            f"Epoch {epoch + 1}: "
            f"Loss={loss.item():.6f}"
        )

# 14. Проверяем Base weights

In [ ]:
print(
    "Base weight unchanged:",
    torch.allclose(
        base_weight_before,
        lora_layer.base.weight,
    )
)

print(
    "Base bias unchanged:",
    torch.allclose(
        base_bias_before,
        lora_layer.base.bias,
    )
)

# 15. Проверяем Adapter

In [ ]:
print(
    "A changed:",
    not torch.allclose(
        A_before,
        lora_layer.lora_A,
    )
)

print(
    "B changed:",
    not torch.allclose(
        B_before,
        lora_layer.lora_B,
    )
)

# 16. Effective Delta W

In [ ]:
with torch.no_grad():
    delta_w = (
        lora_layer.lora_B
        @ lora_layer.lora_A
    ) * lora_layer.scale

print("Delta W shape:", delta_w.shape)
print(
    "Delta W norm:",
    torch.linalg.vector_norm(
        delta_w
    ).item(),
)

# 17. Влияние rank

In [ ]:
for rank in [1, 2, 4, 8]:
    adapter_params = (
        rank * INPUT_DIM
        + OUTPUT_DIM * rank
    )

    print(
        f"rank={rank}: "
        f"{adapter_params} LoRA parameters"
    )

# 18. Формат Instruction Dataset

In [ ]:
instruction_dataset = [
    {
        "messages": [
            {
                "role": "user",
                "content": "Что такое флет на валютном рынке?",
            },
            {
                "role": "assistant",
                "content": (
                    "Флет — состояние рынка, при котором "
                    "цена длительное время движется "
                    "в ограниченном диапазоне без "
                    "устойчивого направленного тренда."
                ),
            },
        ]
    },
    {
        "messages": [
            {
                "role": "user",
                "content": "Что такое тренд?",
            },
            {
                "role": "assistant",
                "content": (
                    "Тренд — устойчивое направленное "
                    "движение цены на выбранном "
                    "временном интервале."
                ),
            },
        ]
    },
]

instruction_dataset

# 19. Train / Validation

In [ ]:
split_index = int(
    len(instruction_dataset) * 0.8
)

train_data = (
    instruction_dataset[:split_index]
)

validation_data = (
    instruction_dataset[split_index:]
)

print("Train:", len(train_data))
print("Validation:", len(validation_data))

Для настоящего Dataset из двух примеров это разделение слишком маленькое. Здесь оно только показывает принцип. Для реальной оценки нужно больше независимых примеров.


# 20. QLoRA — архитектурная схема

In [ ]:
qlora_pipeline = [
    "Base Model",
    "Quantized Base Weights",
    "Frozen Base Model",
    "Trainable LoRA Adapters",
    "Forward",
    "Loss",
    "Backward",
    "Update LoRA only",
]

for step in qlora_pipeline:
    print("→", step)

# 21. 📌 Что нужно запомнить

```text
Full Fine-tuning:
обучается большая часть weights

LoRA:
Base frozen
+
маленькие A/B обучаются

QLoRA:
Quantized Base
+
LoRA Adapters
```


# 22. 🧩 Эксперименты

Попробуй:

- `rank=1`;
- `rank=2`;
- `rank=8`;
- изменить `alpha`;
- изменить Learning Rate;
- проверить Trainable %;
- убедиться, что Base weights остаются неизменными.


# 23. ➡️ Следующая глава

# Глава 13. Итоговый локальный AI-ассистент
